# 02 — ROI Detection: Faster R-CNN (torchvision)

Detect the "Reading Digit" window (ROI) using Faster R-CNN ResNet50-FPN from torchvision.  
Two independent experiments: utility-meter dataset (COCO format, 45 ROI images) and waterMeterDataset (polygon ROI, 1244 images).

In [ ]:
import sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if not IN_COLAB:
    try:
        from google.colab import drive
        IN_COLAB = True
    except ImportError:
        pass

if IN_COLAB:
    from google.colab import drive, userdata
    drive.mount("/content/drive")
    token = userdata.get("GITHUB_TOKEN", "")
    base = f"https://{token}@github.com" if token else "https://github.com"
    if not Path("/content/WaterMeterCV").exists():
        subprocess.run(["git", "clone", f"{base}/UrranQx/WaterMeterCV.git", "/content/WaterMeterCV"], check=True)
    BRANCH = "feature/roi-detection"
    subprocess.run(["git", "-C", "/content/WaterMeterCV", "checkout", BRANCH], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "rapidfuzz", "shapely"], check=True)
    ROOT         = Path("/content/WaterMeterCV")
    DATA_ROOT    = Path("/content/drive/MyDrive/WaterMeterCV/WaterMetricsDATA")
    WEIGHTS_BASE = Path("/content/drive/MyDrive/WaterMeterCV/weights")
    RESULTS_DIR  = Path("/content/drive/MyDrive/WaterMeterCV/results")
    WORKERS = 2
else:
    ROOT         = Path("../..").resolve()
    DATA_ROOT    = ROOT / "WaterMetricsDATA"
    WEIGHTS_BASE = ROOT / "models/weights"
    RESULTS_DIR  = ROOT / "results"
    WORKERS = 0

sys.path.insert(0, str(ROOT))
WEIGHTS_BASE.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import json, time, csv
import torch, cv2, numpy as np
from datetime import datetime

print(f"ROOT: {ROOT}")
print(f"torch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
COCO_ROOT = DATA_ROOT / "utility-meter-reading-dataset-for-automatic-reading-yolo.v4i.coco"
WM_PATH   = DATA_ROOT / "waterMeterDataset/WaterMeters"

WEIGHTS_DIR = WEIGHTS_BASE / "roi_faster_rcnn"
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)

DEVICE       = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EPOCHS_UM    = 20
EPOCHS_WM    = 15   # loss plateaus by ~15 epochs; StepLR fires at epoch 7
BATCH_SIZE   = 4    # conservative for GTX 1660 Ti 6 GB; increase to 8 only if no OOM
LR           = 0.005
SCORE_THRESH = 0.5

RUN_NAME = f"frcnn_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
print(f"Device: {DEVICE}")
print(f"Run: {RUN_NAME}")

In [ ]:
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF
from torchvision.models.detection import fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
try:
    from tqdm import tqdm as _tqdm
    HAS_TQDM = True
except ImportError:
    HAS_TQDM = False

from models.data.unified_loader import load_water_meter_dataset_split
from models.data.roi_dataset import polygon_to_bbox
from models.metrics.evaluation import compute_iou_bbox


class UMRoiDataset(Dataset):
    """Utility-meter COCO split filtered to 'Reading Digit' (category 11)."""
    def __init__(self, split: str):
        json_path = COCO_ROOT / split / "_annotations.coco.json"
        with open(json_path) as f:
            coco = json.load(f)
        roi_cat_id = next(c["id"] for c in coco["categories"] if c["name"] == "Reading Digit")
        img_lookup = {img["id"]: img for img in coco["images"]}
        anns_by_img = {}
        for ann in coco["annotations"]:
            if ann["category_id"] == roi_cat_id:
                anns_by_img.setdefault(ann["image_id"], []).append(ann)
        self.samples = []
        for img_id, anns in anns_by_img.items():
            info = img_lookup[img_id]
            img_path = COCO_ROOT / split / info["file_name"]
            x, y, w, h = anns[0]["bbox"]  # XYWH absolute
            self.samples.append((img_path, [x, y, x + w, y + h]))  # XYXY

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        img_path, box = self.samples[idx]
        img = cv2.imread(str(img_path))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        tensor = TF.to_tensor(img)
        target = {
            "boxes": torch.tensor([box], dtype=torch.float32),
            "labels": torch.ones(1, dtype=torch.int64),
        }
        return tensor, target


class WMRoiDataset(Dataset):
    """waterMeterDataset samples with roi_polygon."""
    def __init__(self, samples):
        self.samples = [s for s in samples if s.roi_polygon is not None]

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        img = cv2.imread(str(s.image_path))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]
        tensor = TF.to_tensor(img)
        cx, cy, bw, bh = polygon_to_bbox(s.roi_polygon)
        x1, y1 = (cx - bw / 2) * w, (cy - bh / 2) * h
        x2, y2 = (cx + bw / 2) * w, (cy + bh / 2) * h
        target = {
            "boxes": torch.tensor([[x1, y1, x2, y2]], dtype=torch.float32),
            "labels": torch.ones(1, dtype=torch.int64),
        }
        return tensor, target


def collate_fn(batch):
    return tuple(zip(*batch))


def build_model(num_classes=2):
    model = fasterrcnn_resnet50_fpn(weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT)
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    return model


def train_model(model, loader, epochs, device, best_path=None):
    """Train Faster R-CNN. Saves best checkpoint to best_path if provided.

    SGD+momentum is used (not Adam) — standard for Faster R-CNN per the original
    paper and torchvision examples; Adam converges faster but typically gives
    lower final mAP on detection tasks.
    """
    model.to(device)
    optimizer = torch.optim.SGD(
        [p for p in model.parameters() if p.requires_grad],
        lr=LR, momentum=0.9, weight_decay=1e-4
    )
    scheduler = torch.optim.lr_scheduler.StepLR(
        optimizer, step_size=max(1, epochs // 2), gamma=0.1
    )
    best_loss = float("inf")

    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        it = _tqdm(loader, desc=f"epoch {epoch+1}/{epochs}", leave=False) if HAS_TQDM else loader
        for images, targets in it:
            images  = [img.to(device) for img in images]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
            loss_dict = model(images, targets)
            loss = sum(loss_dict.values())
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            if HAS_TQDM:
                it.set_postfix(loss=f"{loss.item():.4f}")
        scheduler.step()

        avg_loss = total_loss / len(loader)
        if best_path and avg_loss < best_loss:
            best_loss = avg_loss
            torch.save(model.state_dict(), best_path)

        if (epoch + 1) % 5 == 0 or epoch == 0:
            marker = " *" if best_path and avg_loss == best_loss else ""
            print(f"  epoch {epoch+1}/{epochs}  loss={avg_loss:.4f}{marker}")

    return model


@torch.no_grad()
def eval_model(model, dataset, device):
    model.eval()
    ious, times = [], []
    for i in range(len(dataset)):
        tensor, target = dataset[i]
        img_t = tensor.unsqueeze(0).to(device)
        h, w = tensor.shape[1], tensor.shape[2]

        t0 = time.perf_counter()
        preds = model(img_t)[0]
        times.append((time.perf_counter() - t0) * 1000)

        gt_box = target["boxes"][0].tolist()  # XYXY absolute
        gt_cx = (gt_box[0] + gt_box[2]) / (2 * w)
        gt_cy = (gt_box[1] + gt_box[3]) / (2 * h)
        gt_bw = (gt_box[2] - gt_box[0]) / w
        gt_bh = (gt_box[3] - gt_box[1]) / h

        mask = preds["scores"] >= SCORE_THRESH
        if mask.any():
            b = preds["boxes"][mask][0].tolist()
            pr_cx = (b[0] + b[2]) / (2 * w)
            pr_cy = (b[1] + b[3]) / (2 * h)
            pr_bw = (b[2] - b[0]) / w
            pr_bh = (b[3] - b[1]) / h
            ious.append(compute_iou_bbox((pr_cx, pr_cy, pr_bw, pr_bh), (gt_cx, gt_cy, gt_bw, gt_bh)))
        else:
            ious.append(0.0)

    mean_iou = float(np.mean(ious)) if ious else 0.0
    det_rate  = sum(1 for v in ious if v > 0) / len(ious) if ious else 0.0
    avg_ms    = float(np.mean(times)) if times else 0.0
    return mean_iou, det_rate, avg_ms, ious


print("Helpers loaded.")

## Experiment 1: utility-meter dataset

In [ ]:
um_train_ds = UMRoiDataset("train")
um_test_ds  = UMRoiDataset("test")
um_train_loader = DataLoader(um_train_ds, batch_size=BATCH_SIZE, shuffle=True,
                             num_workers=WORKERS, collate_fn=collate_fn)
print(f"UM train: {len(um_train_ds)}, test: {len(um_test_ds)}")

model_um = build_model()
print(f"Training Faster R-CNN on UM ({EPOCHS_UM} epochs)...")
model_um = train_model(model_um, um_train_loader, EPOCHS_UM, DEVICE,
                       best_path=WEIGHTS_DIR / f"um_best_{RUN_NAME}.pth")
torch.save(model_um.state_dict(), WEIGHTS_DIR / f"um_last_{RUN_NAME}.pth")
print("UM training done.")

In [ ]:
um_mean_iou, um_det_rate, um_avg_ms, um_ious = eval_model(model_um, um_test_ds, DEVICE)
print(f"UM — Mean IoU:       {um_mean_iou:.4f}")
print(f"UM — Detection rate: {um_det_rate:.4f} ({sum(1 for v in um_ious if v>0)}/{len(um_ious)})")
print(f"UM — Avg inference:  {um_avg_ms:.1f} ms/image")

## Experiment 2: waterMeterDataset

In [ ]:
wm_train_raw, wm_test_raw = load_water_meter_dataset_split(WM_PATH, train_ratio=0.7, seed=42)
wm_train_ds = WMRoiDataset(wm_train_raw)
wm_test_ds  = WMRoiDataset(wm_test_raw)
wm_train_loader = DataLoader(wm_train_ds, batch_size=BATCH_SIZE, shuffle=True,
                             num_workers=WORKERS, collate_fn=collate_fn)
print(f"WM train: {len(wm_train_ds)}, test: {len(wm_test_ds)}")

model_wm = build_model()
print(f"Training Faster R-CNN on WM ({EPOCHS_WM} epochs)...")
model_wm = train_model(model_wm, wm_train_loader, EPOCHS_WM, DEVICE,
                       best_path=WEIGHTS_DIR / f"wm_best_{RUN_NAME}.pth")
torch.save(model_wm.state_dict(), WEIGHTS_DIR / f"wm_last_{RUN_NAME}.pth")
print("WM training done.")

In [ ]:
wm_mean_iou, wm_det_rate, wm_avg_ms, wm_ious = eval_model(model_wm, wm_test_ds, DEVICE)
print(f"WM — Mean IoU:       {wm_mean_iou:.4f}")
print(f"WM — Detection rate: {wm_det_rate:.4f} ({sum(1 for v in wm_ious if v>0)}/{len(wm_ious)})")
print(f"WM — Avg inference:  {wm_avg_ms:.1f} ms/image")

## Predictions

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 10))

@torch.no_grad()
def draw_row(axes_row, dataset, model, ious_list):
    model.eval()
    for i, ax in enumerate(axes_row):
        if i >= len(dataset):
            ax.axis("off"); continue
        tensor, target = dataset[i]
        img_np = (tensor.permute(1, 2, 0).numpy() * 255).astype(np.uint8).copy()

        gt = target["boxes"][0].tolist()
        cv2.rectangle(img_np, (int(gt[0]), int(gt[1])), (int(gt[2]), int(gt[3])), (0, 200, 0), 2)

        preds = model(tensor.unsqueeze(0).to(DEVICE))[0]
        mask  = preds["scores"] >= SCORE_THRESH
        if mask.any():
            b = preds["boxes"][mask][0].tolist()
            cv2.rectangle(img_np, (int(b[0]), int(b[1])), (int(b[2]), int(b[3])), (200, 0, 0), 2)

        ax.imshow(img_np)
        ax.set_title(f"IoU={ious_list[i]:.2f}" if i < len(ious_list) else "", fontsize=10)
        ax.axis("off")

draw_row(axes[0], um_test_ds, model_um, um_ious)
draw_row(axes[1], wm_test_ds, model_wm, wm_ious)
axes[0][0].set_ylabel("UM", fontsize=11)
axes[1][0].set_ylabel("WM", fontsize=11)
plt.suptitle("Faster R-CNN ROI — Green=GT, Red=Pred", fontsize=14)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "roi_faster_rcnn_predictions.png", dpi=150)
plt.close()
print("Saved roi_faster_rcnn_predictions.png")

## Comparison

In [ ]:
print(f"{'='*60}")
print(f"{'Metric':<25} {'utility-meter':>15} {'waterMeter':>15}")
print(f"{'='*60}")
print(f"{'Mean IoU':<25} {um_mean_iou:>15.4f} {wm_mean_iou:>15.4f}")
print(f"{'Detection rate':<25} {um_det_rate:>15.4f} {wm_det_rate:>15.4f}")
print(f"{'Avg inference (ms)':<25} {um_avg_ms:>15.1f} {wm_avg_ms:>15.1f}")
print(f"{'N test':<25} {len(um_test_ds):>15d} {len(wm_test_ds):>15d}")
print(f"{'='*60}")

## Save Results

In [ ]:
metrics = {
    "method": "faster_rcnn_torchvision",
    "utility_meter": {
        "mean_iou": round(um_mean_iou, 4),
        "detection_rate": round(um_det_rate, 4),
        "avg_inference_ms": round(um_avg_ms, 1),
        "n_train": len(um_train_ds),
        "n_test": len(um_test_ds),
    },
    "water_meter": {
        "mean_iou": round(wm_mean_iou, 4),
        "detection_rate": round(wm_det_rate, 4),
        "avg_inference_ms": round(wm_avg_ms, 1),
        "n_train": len(wm_train_ds),
        "n_test": len(wm_test_ds),
    },
    "config": {
        "epochs_um": EPOCHS_UM,
        "epochs_wm": EPOCHS_WM,
        "batch_size": BATCH_SIZE,
        "lr": LR,
        "backbone": "resnet50_fpn",
    },
    "run_date": datetime.now().isoformat(),
}

with open(RESULTS_DIR / "roi_faster_rcnn_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

csv_path = RESULTS_DIR / "roi_comparison.csv"
csv_exists = csv_path.exists()
with open(csv_path, "a", newline="") as f:
    writer = csv.writer(f)
    if not csv_exists:
        writer.writerow(["method","um_mean_iou","um_detection_rate","um_inference_ms",
                         "wm_mean_iou","wm_detection_rate","wm_inference_ms","run_date"])
    writer.writerow(["faster_rcnn_torchvision",
                     round(um_mean_iou,4), round(um_det_rate,4), round(um_avg_ms,1),
                     round(wm_mean_iou,4), round(wm_det_rate,4), round(wm_avg_ms,1),
                     datetime.now().strftime("%Y-%m-%d %H:%M")])

print("Saved roi_faster_rcnn_metrics.json")
print("Saved roi_comparison.csv")

## Conclusions

*(Fill after running)*

- **Faster R-CNN torchvision (utility-meter):** Mean IoU=..., Detection rate=...
- **Faster R-CNN torchvision (waterMeter):** Mean IoU=..., Detection rate=...
- **Inference:** ... ms/image
- **Note:** UM dataset too small (45 train images) — results unreliable by design.